# PlacementIQ — Notebook 05: Model Explainability & Interpretability

**Objective:**
1. Load the finalized, trained placement prediction pipeline.
2. Analyze global feature importances across the entire feature space.
3. Implement SHAP (TreeExplainer) to compute feature contribution attributions.
4. Generate Global SHAP Summary (beeswarm) and Feature Importance bar charts.
5. Demonstrate local individual explainability for sample placed vs unplaced profiles.
6. Connect explainability outputs to the Skill-Gap Analysis engine.

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import load_model, predict_placement, compute_skill_gaps, simulate_what_if
from explainability import (
    _get_transformed_data_and_names, build_shap_explainer,
    compute_shap_values, plot_shap_summary, plot_shap_importance,
    explain_individual, SHAP_AVAILABLE
)

sns.set_theme(style="whitegrid")
print(f"SHAP Library Available: {SHAP_AVAILABLE}")

## 2. Load Final Pipeline & Holdout Data

In [ ]:
model_path = Path('../models/placement_model.joblib')
pipeline = joblib.load(model_path)

with open('../models/model_metadata.json') as f:
    metadata = json.load(f)

print(f"Loaded Model      : {metadata['model_name']}")
print(f"Optimal Threshold : {metadata['classification_threshold']}")

X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')['placement_status']
print(f"Holdout samples   : {X_test.shape}")

## 3. Global Feature Importance (Tree-Based)

In [ ]:
X_transformed, feature_names = _get_transformed_data_and_names(pipeline, X_test.head(100))
classifier = pipeline.named_steps['classifier']

if hasattr(classifier, 'feature_importances_'):
    imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': classifier.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=imp_df.head(15), x='Importance', y='Feature', palette='Blues_r')
    plt.title(f"Top 15 Feature Importances — {metadata['model_name']}", fontweight='bold')
    plt.xlabel("Relative Importance Score")
    plt.tight_layout()
    plt.show()

## 4. Local Individual Prediction & Skill-Gap Analysis

In [ ]:
sample_student = X_test.iloc[0].to_dict()
print("Sample Student Attributes:")
for k, v in list(sample_student.items())[:8]:
    print(f"  {k}: {v}")

pred = predict_placement(pipeline, sample_student, threshold=metadata['classification_threshold'])
print(f"\nPrediction Result: {pred['placement_label']} (Probability: {pred['placement_probability']:.1%})")

gaps = compute_skill_gaps(sample_student)
print(f"\nSkill-Gap Areas Identified: {gaps['total_gaps_identified']}")
for g in gaps['gaps'][:3]:
    print(f"  • {g['skill'].upper()}: current={g['student_value']}, benchmark={g['benchmark']} -> {g['recommendation']}")

## 5. What-If Scenario Simulation

In [ ]:
modifications = {
    'coding_skills': min(10.0, sample_student['coding_skills'] + 2.0),
    'dsa_score': min(10.0, sample_student['dsa_score'] + 2.0),
    'internships': sample_student['internships'] + 1
}

sim = simulate_what_if(pipeline, sample_student, modifications, threshold=metadata['classification_threshold'])
print(f"Original Probability : {sim['original']['probability']:.1%}")
print(f"Modified Probability : {sim['modified']['probability']:.1%}")
print(f"Net Delta            : {sim['probability_change_pct']:+.2f} percentage points")